# FS-DualNet v2 on Kaggle: patch-level frequency-spatial fusion (all experiments, one commit)
**Session options:** Accelerator = **GPU T4 x2**, **Internet = On** (downloads CLIP ViT-B/16 and DeepFakeFace).
**+ Add Input:** `fs-dualnet-code` (**latest version**) and `diffusionforensics`.
**Run:** Save Version -> **Save & Run All (Commit)**. Estimated 5-8 h for everything. After `BUDGET_H` = 11 h no new epoch or run starts (Kaggle stops at 12 h).
If the last cell says NOT FINISHED: Save Version again with **this notebook's own output** added as input; finished runs are skipped.

### Protocol (fixed before looking at any test number)
| experiment | train | early stopping | DEV: threshold + choosing configurations | FINAL TEST (unseen) |
|---|---|---|---|---|
| A. cross-generator | celebahq `sdv2` | celebahq `sdv2` val | celebahq `if` (`dev_if`) | `dalle2`, `midjourney` (`sdv2` test = in-domain reference) |
| B. cross-dataset | DFF `inpainting` | DFF `inpainting` val | DFF `text2img` (`dev_text2img`) | celebahq `cb_sdv2`, `cb_dalle2`, `cb_if`, `cb_midjourney`; DFF `insight` (face swap) separately as `x_insight` |

**Runs:** `v2` (proposed), `v2_spatial` (frozen-CLIP branch alone), `v2_freq` (residual-DCT branch alone): 3 seeds each, both experiments.
**Ablations (seed 42):** `v2_nodeg` (no degradation augmentation), `v2_raw` (DCT of the raw image), `v2_nomd` (no modality dropout / auxiliary heads), `v2_effnet` (fine-tuned EfficientNet-B4 instead of frozen CLIP).
Report every run. ACC@oracle uses test labels: an upper bound, never the reported accuracy.

In [ ]:
import time
BUDGET_H = 11    # hours this commit may use (Kaggle stops a session at 12 h). Lower it if less weekly GPU quota is left.
T_START = time.time()
DEADLINE = T_START + BUDGET_H * 3600   # no new epoch / run starts after this; everything is resumable
!nvidia-smi
import torch
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU: Settings -> Accelerator -> GPU T4 x2"
print(torch.cuda.get_device_name(0))

## 1. Find code, data and earlier results

In [ ]:
import os, sys, shutil, subprocess, glob

INPUT, WORK = '/kaggle/input', '/kaggle/working'

def scan(root, maxdepth=7):
    # one pass over the inputs: folders by name, folders holding train_eval.py, zip files
    dirs, codes, zips = {}, [], []
    base = root.rstrip('/').count('/')
    for dp, dns, fns in os.walk(root):
        for d in dns:
            dirs.setdefault(d.lower(), []).append(os.path.join(dp, d))
        if 'train_eval.py' in fns:
            codes.append(dp)
        zips += [os.path.join(dp, f) for f in fns if f.lower().endswith('.zip')]
        if dp.count('/') - base >= maxdepth:
            dns[:] = []
    return dirs, codes, zips

def show_tree(root, depth=4, limit=60):
    base, n = root.rstrip('/').count('/'), 0
    for dp, dns, fns in os.walk(root):
        k = dp.count('/') - base
        print('  ' * k + (os.path.basename(dp) or dp) + '/')
        for f in sorted(fns)[:5]:
            print('  ' * (k + 1) + f)
        n += 1
        if k >= depth - 1:
            dns[:] = []
        if n >= limit:
            print('  ...'); return

def fail(msg):
    print('What is under', INPUT, ':'); show_tree(INPUT)
    raise AssertionError(msg)

def is_v2(code_dir):
    try:
        return 'FSDualNetV2' in open(os.path.join(code_dir, 'fs_dualnet.py')).read()
    except OSError:
        return False

DIRS, CODES, ZIPS = scan(INPUT)

# 1) code = dataset 'fs-dualnet-code' (the version that contains FSDualNetV2)
CODE = next((c for c in CODES if is_v2(c)), None)
for z in ([] if CODE else [z for z in ZIPS if 'dualnet' in os.path.basename(z).lower()]):
    d = f'/tmp/code{len(z)}'
    subprocess.run(['unzip', '-q', '-o', z, '-d', d], check=True)
    CODE = next((c for c in scan(d)[1] if is_v2(c)), None)
    if CODE:
        break
if CODE is None:
    fail("v2 code not found: upload the NEW fs_dualnet.zip as a New Version of 'fs-dualnet-code', then in this notebook "
         "Input -> fs-dualnet-code -> update to the latest version")
shutil.copytree(CODE, f'{WORK}/fs_dualnet', dirs_exist_ok=True)
os.chdir(f'{WORK}/fs_dualnet')

# 2) data = dataset 'diffusionforensics' (Images/ and dire/)
imgs = [d for d in DIRS.get('images', []) if os.path.isdir(f'{d}/test') or os.path.isdir(f'{d}/train')]
if not imgs:
    for z in [z for z in ZIPS if 'diffusion' in os.path.basename(z).lower()]:
        subprocess.run(['unzip', '-q', '-o', z, '-d', '/tmp/raw'], check=True)
        imgs = [d for d in scan('/tmp/raw')[0].get('images', []) if os.path.isdir(f'{d}/test')]
        if imgs:
            break
if not imgs:
    fail("data not found: + Add Input -> 'diffusionforensics'")
RAW_ROOT = os.path.dirname(imgs[0])

# 3) results of an earlier commit of THIS notebook -> resume
R = f'{WORK}/v2_results'
prev = DIRS.get('v2_results', [])
if prev and not os.path.exists(R):
    shutil.copytree(prev[0], R)
    print('RESUMING from', prev[0], '->', sorted(os.listdir(R)))
os.makedirs(R, exist_ok=True)

print('code :', CODE)
print('data :', RAW_ROOT)
print('out  :', R)

In [ ]:
# transformers (CLIP) + one-time CLIP download + smoke test of every model variant
import importlib.util
if importlib.util.find_spec('transformers') is None:
    !pip install -q transformers
r = subprocess.run([sys.executable, '-c', "from transformers import CLIPVisionModel as M; M.from_pretrained('openai/clip-vit-base-patch16')"],
                   capture_output=True, text=True)
if r.returncode:
    print(r.stderr[-2000:])
assert r.returncode == 0, 'CLIP download failed: Settings -> Internet must be On'
print('CLIP ViT-B/16 downloaded (cached for all runs)')
!python fs_dualnet.py

## 2. Experiment plan + data (cached at 256x256, links only)

In [ ]:
EPOCHS = 5   # SAME for every v2 run (light heads on a frozen backbone)
MAIN3 = ['v2', 'v2_spatial', 'v2_freq']
PLAN = ([('cb', 42, t) for t in MAIN3] + [('dff', 42, t) for t in MAIN3]                 # main results first
        + [(e, s, t) for s in (1, 2) for e in ('cb', 'dff') for t in MAIN3]               # seeds 1 and 2
        + [('cb', 42, t) for t in ['v2_nodeg', 'v2_raw', 'v2_nomd', 'v2_effnet']]         # ablations
        + [('dff', 42, 'v2_nodeg')])
done = lambda e, s, t: os.path.exists(f'{R}/{e}_s{s}/results_{t}.json')
left = [p for p in PLAN if not done(*p)]
print(f'{len(PLAN) - len(left)} of {len(PLAN)} runs already finished; to do:', left)

def sh(cmd):
    print('$', cmd if len(cmd) < 300 else cmd[:300] + ' ...', flush=True)
    get_ipython().system(cmd)

In [ ]:
# A) DiffusionForensics celebahq (needed by both experiments: its test sets are experiment B's final test)
sh(f'python prep_data.py --root {RAW_ROOT} --out /tmp/data_cb')
sh('python cache_resize.py --src /tmp/data_cb --dst /tmp/c256/cb --size 256 --procs 4')

In [ ]:
# B) DeepFakeFace from Hugging Face (public), pair-aware split; skipped when no DFF run is left
if any(e == 'dff' for e, _, _ in left):
    os.makedirs('/tmp/dff', exist_ok=True)
    for n in ['wiki', 'inpainting', 'text2img', 'insight']:
        if os.path.isdir(f'/tmp/dff/{n}') and os.listdir(f'/tmp/dff/{n}'):
            continue
        url = f'https://huggingface.co/datasets/OpenRL/DeepFakeFace/resolve/main/{n}.zip'
        try:
            subprocess.run(['curl', '-fL', '--retry', '3', '-sS', '-o', f'/tmp/dff/{n}.zip', url], check=True)
        except subprocess.CalledProcessError:
            raise SystemExit('download failed: Settings -> Internet must be ON')
        os.makedirs(f'/tmp/dff/{n}', exist_ok=True)
        subprocess.run(['unzip', '-q', '-o', f'/tmp/dff/{n}.zip', '-d', f'/tmp/dff/{n}'], check=True)
        os.remove(f'/tmp/dff/{n}.zip')
        print(n, ': done', flush=True)
    sh('python prep_dff.py --root /tmp/dff --out /tmp/data_dff')
    for s in ['train/inpainting', 'val/inpainting', 'test/text2img', 'test/insight']:
        sh(f'python cache_resize.py --src /tmp/data_dff/{s} --dst /tmp/c256/dff/{s} --size 256 --procs 4')
else:
    print('all DFF runs finished: DFF download skipped')
print(subprocess.run('df -h /tmp /kaggle/working | cat', shell=True, capture_output=True, text=True).stdout)

In [ ]:
C, D = '/tmp/c256/cb', '/tmp/c256/dff'
EXP = {
    'cb':  dict(data=C, train_gen='sdv2', dev=f'dev_if={C}/test/if',
                tests=' '.join(f'{g}={C}/test/{g}' for g in ['dalle2', 'midjourney', 'sdv2']),
                unseen='dalle2 midjourney'),
    'dff': dict(data=D, train_gen='inpainting', dev=f'dev_text2img={D}/test/text2img',
                tests=' '.join([f'x_insight={D}/test/insight'] + [f'cb_{g}={C}/test/{g}' for g in ['sdv2', 'dalle2', 'if', 'midjourney']]),
                unseen='cb_sdv2 cb_dalle2 cb_if cb_midjourney'),
}

def run(exp, seed, runs):
    e = EXP[exp]
    sh(f"python run_all.py --data {e['data']} --train_gen {e['train_gen']} --dev {e['dev']} --tests {e['tests']} "
       f"--out_dir {R}/{exp}_s{seed} --seed {seed} --runs {' '.join(runs)} --epochs {EPOCHS} --workers 4 --bs 32 --deadline {DEADLINE}")

## 3. All runs, in priority order (main results -> seeds 1, 2 -> ablations)
Interrupted? Run the cells above and this cell again: finished runs are skipped, an interrupted run resumes from its last epoch.

In [ ]:
from itertools import groupby
for (e, s), grp in groupby(PLAN, key=lambda p: (p[0], p[1])):
    runs = [t for _, _, t in grp if not done(e, s, t)]
    if not runs:
        continue
    if time.time() > DEADLINE - 600:
        print('TIME BUDGET REACHED: nothing new is started'); break
    print(f'\n######## experiment {e}, seed {s}: {runs}  ({(time.time() - T_START) / 3600:.2f} h elapsed)', flush=True)
    run(e, s, runs)

## 4. Speed, tables, claim check

In [ ]:
sh(f'python benchmark_latency.py --out {R}/latency.json')
for e in EXP:
    if os.path.isdir(f'{R}/{e}_s42'):
        shutil.copy(f'{R}/latency.json', f'{R}/{e}_s42/latency.json')

In [ ]:
# results table, mean over seeds with paired per-seed comparisons, claim check
for e in EXP:
    u = EXP[e]['unseen']
    d42 = f'{R}/{e}_s42'
    if not os.path.exists(f'{d42}/results_v2.json'):
        continue
    print(f'\n==================== experiment {e} (unseen = {u}) ====================')
    sh(f'python make_table.py --dir {d42} --unseen {u}')
    dirs = [f'{R}/{e}_s{s}' for s in (42, 1, 2) if os.path.exists(f'{R}/{e}_s{s}/results_v2.json')]
    if len(dirs) >= 2:
        sh(f"python aggregate_seeds.py --dirs {' '.join(dirs)} --main v2 --unseen {u} --out {d42}")
    sh(f'python claims.py --dir {d42} --main v2 --baseline v2_spatial --unseen {u}')

## 5. Explanations (v2, experiment A): input | token Grad-CAM | patch evidence P(fake) | modality map (red = frequency, blue = spatial)
Fully synthetic faces have no region-level ground truth: these maps show decision-relevant regions, not forged regions.

In [ ]:
from IPython.display import Image as IPImage, display
ck = f'{R}/cb_s42/v2.pt'
if os.path.exists(ck):
    for g in ['dalle2', 'midjourney']:
        for kind in ['fake', 'real']:
            img = sorted(glob.glob(f'{C}/test/{g}/{kind}/*'))[0]
            out = f'{R}/cb_s42/explain_{g}_{kind}.png'
            sh(f'python explain.py --ckpt {ck} --image {img} --out {out}')
            display(IPImage(out))

## 6. Status + one small summary file to download (tables, results, predictions, figures; no checkpoints)

In [ ]:
S = '/tmp/v2_summary'
shutil.rmtree(S, ignore_errors=True)
for f in glob.glob(f'{R}/**/*', recursive=True):
    if os.path.isfile(f) and f.endswith(('.md', '.json', '.png', '.npz')):
        dst = os.path.join(S, os.path.relpath(f, R))
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy(f, dst)
shutil.make_archive(f'{WORK}/v2_summary', 'zip', S)
left = [p for p in PLAN if not done(*p)]
print(f'elapsed {(time.time() - T_START) / 3600:.2f} h | Output tab -> v2_summary.zip ({os.path.getsize(f"{WORK}/v2_summary.zip") // 1024} KB)')
print('ALL RUNS FINISHED' if not left else
      f'NOT FINISHED ({len(left)} runs left: {left})\n-> Save Version again with THIS notebook output added as input (+ Add Input -> Your Work)')